In [ ]:
import os
import xlwings as xw

cptu_path = r"C:\Users\jdr\Desktop\CTPU_Tåjeodden\Test"
ocr_value = "OCR"  # Set your desired value for Generelt!C31

p_values = [0.5, 1, 2, 4, 6, 10, 12, 15]
p_cells = [f"P{r}" for r in range(9, 17)]
q_cells = [f"Q{r}" for r in range(9, 17)]

def q_formula(p_cell_ref):
    return f"=1+5/XLOOKUP({p_cell_ref},Beregn!A20:A2000,Beregn!U20:U2000)"
    #return f"=IF(ISNA(1+5/XLOOKUP({p_cell_ref},Beregn!A20:A2000,Beregn!U20:U2000)),\"\",(1+5/XLOOKUP({p_cell_ref},Beregn!A20:A2000,Beregn!U20:U2000)))"
for fname in os.listdir(cptu_path):
    if not fname.lower().endswith(".xlsm") or fname.startswith("~$"):
        continue
    fpath = os.path.join(cptu_path, fname)
    print(f"Processing: {fname}")
    app = xw.App(visible=False)
    try:
        wb = xw.Book(fpath)
        # Generelt!C31
        try:
            sht_gen = wb.sheets["Generelt"]
            sht_gen.range("C31").value = ocr_value
        except Exception:
            print(f"Skipping {fname}: sheet 'Generelt' not found")
        # 7.OCR
        try:
            sht_ocr = wb.sheets["7.OCR"]
            for cell_addr, val in zip(p_cells, p_values):
                sht_ocr.range(cell_addr).value = val
            for q_addr, p_addr in zip(q_cells, p_cells):
                formula = q_formula(p_addr)
                try:
                    sht_ocr.range(q_addr).formula = formula
                    # Check if the result is #N/A or None
                    cell_value = sht_ocr.range(q_addr).value
                    if cell_value is None or str(cell_value).strip().upper() == "#N/A":
                        sht_ocr.range(q_addr).value = ""
                        sht_ocr.range(p_addr).value = ""
                except Exception as e:
                    print(f"Failed to write formula to {q_addr}: {e}")
            # Set Q3 value
            sht_ocr.range("Q3").value = "OCR4 Egendefinert"
        except Exception as e:
            print(f"Error in {fname}: {e}")
            print(f"Skipping {fname}: sheet '7.OCR' not found")
        # 8.Cu
        try:
            sht_cu = wb.sheets["8.Cu"]
            sht_cu.range("AG5").value = 0.32
            sht_cu.range("AG6").value = 0.7
        except Exception:
            print(f"Skipping {fname}: sheet '8.Cu' not found")
        wb.save()
        wb.close()
        print(f"Updated: {fname}")
    except Exception as e:
        print(f"Failed {fname}: {e}")
    finally:
        app.quit()

Processing: CPT-1.xlsm
Updated: CPT-1.xlsm
Processing: CPTU_2.xlsm
Updated: CPTU_2.xlsm


In [ ]:
# Samleplot SHANSEP POP
import os
import xlwings as xw

cptu_path = r"C:\Users\jdr\OneDriveMulti\SHAREPOINT_Prosjekter\P-10266411-01 Sjøfylling Tåjeodden, Slemmestad - General\CPTU"
output_file = os.path.join(cptu_path, "samleplot_xlwings.xlsx")

# Create a new workbook for the plot
app = xw.App(visible=False)
wb_out = xw.Book()
ws_out = wb_out.sheets[0]
ws_out.name = "Samledata"

start_col = 1

series_info = []

for fname in os.listdir(cptu_path):
    if not fname.lower().endswith(".xlsm") or fname.startswith("~$"):
        continue
    fpath = os.path.join(cptu_path, fname)
    print(f"Processing: {fname}")
    try:
        wb = xw.Book(fpath)
        sht = wb.sheets["Beregn"]
        # Read EC and A columns (Excel columns EC and A, rows 20-10000)
        ec_data = sht.range("EC20:EC10000").value
        a_data = sht.range("A20:A10000").value
        # Flatten if needed
        ec_data = [v for v in ec_data if v is not None]
        a_data = [v for v in a_data if v is not None]
        # Write to output sheet
        ws_out.range((1, start_col)).value = f"{fname}_EC"
        ws_out.range((1, start_col + 1)).value = f"{fname}_A"
        ws_out.range((2, start_col)).value = [[v] for v in ec_data]
        ws_out.range((2, start_col + 1)).value = [[v] for v in a_data]
        # Store info for chart
        last_row = len(ec_data) + 1
        series_info.append({
            "name": fname,
            "x_col": start_col,
            "y_col": start_col + 1,
            "last_row": last_row
        })
        start_col += 2
        wb.close()
    except Exception as e:
        print(f"Failed {fname}: {e}")

# Add chart
chart = ws_out.charts.add()
chart.chart_type = "xy_scatter_lines"
chart.name = "Samleplot fra alle filer"
chart.set_source_data(ws_out.range((1, 1), (2, 1)))  # Dummy, will update below
chart_api = chart.api
if isinstance(chart_api, tuple):
    chart_api = chart_api[0]
chart_com = chart_api.Chart if hasattr(chart_api, "Chart") else chart_api

for info in series_info:
    x_range = ws_out.range((2, info["x_col"]), (info["last_row"], info["x_col"]))
    y_range = ws_out.range((2, info["y_col"]), (info["last_row"], info["y_col"]))
    chart_com.SeriesCollection().NewSeries()
    chart_com.SeriesCollection(chart_com.SeriesCollection().Count).XValues = x_range.api
    chart_com.SeriesCollection(chart_com.SeriesCollection().Count).Values = y_range.api
    chart_com.SeriesCollection(chart_com.SeriesCollection().Count).Name = info["name"]

chart_com.HasTitle = True
chart_com.ChartTitle.Text = "Samleplot fra alle filer"
chart_com.Axes(1).HasTitle = True
chart_com.Axes(1).AxisTitle.Text = "EC"
chart_com.Axes(2).HasTitle = True
chart_com.Axes(2).AxisTitle.Text = "A"

wb_out.save(output_file)
wb_out.close()
app.quit()
print(f"Samleplot lagret som '{output_file}'")

Processing: CPT-1.xlsm
Processing: CPTU_10.xlsm
Processing: CPTU_11.xlsm
Processing: CPTU_12.xlsm
Processing: CPTU_12_utvidet_plot.xlsm
Processing: CPTU_15.xlsm
Processing: CPTU_17.xlsm
Failed CPTU_17.xlsm: (-2147352567, 'Exception occurred.', (0, 'Microsoft Excel', 'Open method of Workbooks class failed', 'xlmain11.chm', 0, -2146827284), None)
Processing: CPTU_18.xlsm
Processing: CPTU_19.xlsm
Processing: CPTU_2.xlsm
Processing: CPTU_20.xlsm
Processing: CPTU_21.xlsm
Processing: CPTU_3.xlsm
Processing: CPTU_4.xlsm
Processing: CPTU_6.xlsm
Processing: CPTU_6_samlet.xlsm
Processing: CPTU_6_samlet_utvidet_plot.xlsm
Processing: CPTU_7.xlsm
Failed CPTU_7.xlsm: (-2147352567, 'Exception occurred.', (0, 'Microsoft Excel', 'Close method of Workbook class failed', 'xlmain11.chm', 0, -2146827284), None)
Processing: CPTU_8.xlsm
Processing: CPTU_9.xlsm
Processing: R-11.xlsm
Processing: ~$CPT-1.xlsm
Failed ~$CPT-1.xlsm: (-2147352567, 'Exception occurred.', (0, 'Microsoft Excel', "Excel cannot open the

AttributeError: 'tuple' object has no attribute 'SeriesCollection'

In [ ]:
# Oppdatere CPTU-filer med Cu-tolkning

import os
import xlwings as xw

cptu_path = r"C:\Users\jdr\Desktop\CTPU_Tåjeodden"
pdf_path = os.path.join(cptu_path, "{}.pdf")
rev_field = "00"
rev_field_01 = "01"
date_str = "19.09.2025"
teg_nr = f"=Generelt!C37&\"-\"&Generelt!C42"
ocr_value = "OCR"  # Set your desired value for Generelt!C31
p_values = [0.5, 1, 2, 4, 6, 10, 12, 15]
p_cells = [f"P{r}" for r in range(9, 17)]
q_cells = [f"Q{r}" for r in range(9, 17)]

# p_values = [0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5, 5.5, 6, 6.5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 25]
# p_cells = [f"P{r}" for r in range(5, len(p_values) + 5)]
# q_cells = [f"Q{r}" for r in range(5, len(p_values) + 5)]


def q_formula(p_cell_ref):
    return f"=1+5/XLOOKUP({p_cell_ref},Beregn!A20:A2000,Beregn!U20:U2000)"

app = xw.App(visible=False)  # Start Excel once

try:
    for fname in os.listdir(cptu_path):
        if not fname.lower().endswith(".xlsm") or fname.startswith("~$"):
            continue
        fpath = os.path.join(cptu_path, fname)
        print(f"Processing: {fname}")
        try:
            wb = xw.Book(fpath)
            # Generelt!C31
            try:
                sht_gen = wb.sheets["Generelt"]
                # OCR value not pc as interpretation
                sht_gen.range("C31").value = ocr_value
                # KS and godkjent initials
                sht_gen.range("C13").value = "ASS"
                sht_gen.range("C14").value = "ERIK-S"
                # Bp_num
                c8_addr = sht_gen.range('C8').address
                # Teg-nr. Cu-plot
                sht_gen.range("C42").value = f"={c8_addr}&\"-500.7\""
                # Rev and date 500.7 aka. Cu tolkning
                sht_gen.range("D42").number_format = "@"
                sht_gen.range("D42").value = rev_field
                sht_gen.range("E42").value = date_str
                # Date and revision fields 3. Profil
                sht_gen.range("E40").value = date_str
                sht_gen.range("D40").value = rev_field_01
                

            except Exception as e:
                print(f"Feil", e)
            # 3.Profil
            try:
                sht_prof = wb.sheets["3.Profil"]
                sht_prof.api.PageSetup.CenterFooter = ""
                sht_prof.api.PageSetup.LeftFooter = ""
                sht_prof.api.PageSetup.RightFooter = ""
                pdf_name = os.path.splitext(fname)[0]
                sht_prof.api.ExportAsFixedFormat(0, pdf_path.format(f"{pdf_name}_Profil"))
            except Exception as e:
                print(f"Feil ved 3.Profil", e)     
            # 7.OCR
            try:
                sht_ocr = wb.sheets["7.OCR"]
                for cell_addr, val in zip(p_cells, p_values):
                    sht_ocr.range(cell_addr).value = val
                for q_addr, p_addr in zip(q_cells, p_cells):
                    formula = q_formula(p_addr)
                    try:
                        sht_ocr.range(q_addr).formula = formula
                        # Check if the result is #N/A or None
                        cell_value = sht_ocr.range(q_addr).value
                        if cell_value is None or str(cell_value).strip().upper() == "#N/A":
                            sht_ocr.range(q_addr).value = ""
                            sht_ocr.range(p_addr).value = ""
                    except Exception as e:
                        print(f"Failed to write formula to {q_addr}: {e}")
                # Set Q3 value
                sht_ocr.range("Q3").value = "OCR4 Egendefinert"
            except Exception as e:
                print(f"Error in {fname}: {e}")
                print(f"Skipping {fname}: sheet '7.OCR' not found")
            # 8.Cu
            try:
                sht_cu = wb.sheets["8.Cu"]
                sht_cu.range("AG5").value = 0.32
                sht_cu.range("AG6").value = 0.7
                sht_cu.range("J51").value = teg_nr
                sht_cu.range("K51").value = ""
                sht_cu.range("K51:M52").unmerge()
                sht_cu.range("J51:M52").merge()
                #sht_cu.range("J51:M52").horizontal_alignment = 'center'
                sht_cu.range("J51:M52").api.HorizontalAlignment = -4108  # xlCenter

                #sht_cu.range("J51").horizontal_alignment = 'center'

                sht_cu.range("P5:Q15").value = ""
                # for chart in sht_cu.charts:
                #     print(chart)
                chart = sht_cu.charts[0]
                chart_obj = chart.api[0]
                # Get the actual Chart COM object
                chart_com = chart_obj.Chart

                target_formula = '=SERIES(Beregn!$EC$16,Beregn!$EC$20:$EC$10000,Beregn!$A$20:$A$10000,11)'  # Example with comma
                hide_formula = '=SERIES(\'8.Cu\'!$P$2:$Q$2,\'8.Cu\'!$Q$5:$Q$120,\'8.Cu\'!$P$5:$P$120,17)'
#Series 10 formula: =SERIES('8.Cu'!$P$2:$Q$2,'8.Cu'!$Q$5:$Q$120,'8.Cu'!$P$5:$P$120,17)

                # for cell_addr, val in zip(p_cells, p_values):
                #     sht_cu.range(cell_addr).value = val
                #     for q_addr, p_addr in zip(q_cells, p_cells):
                #         formula = q_formula(p_addr)
                #         try:
                #             sht_cu.range(q_addr).formula = formula
                #         except Exception as e:
                #             print(f"Failed to write formula to {q_addr}: {e}")

                for i in range(1, chart_com.SeriesCollection().Count + 1):
                    s = chart_com.SeriesCollection(i)
                    if s.Formula == target_formula:
                        s.Format.Line.Weight = 2  # Set line thickness to 2
                    if s.Formula == hide_formula:
                        # Hide the series (make line and markers invisible)
                        s.Format.Line.Visible = False
                        s.IsFiltered = True  # Hide from legend (Excel 2013+)
                # Export as PDF using file name (without extension)
                sht_cu.api.PageSetup.CenterFooter = ""
                sht_cu.api.PageSetup.LeftFooter = ""
                sht_cu.api.PageSetup.RightFooter = ""
                pdf_name = os.path.splitext(fname)[0]
                sht_cu.api.ExportAsFixedFormat(0, pdf_path.format(f"{pdf_name}_Cu"))
            except Exception as e:
                print(f"Error in {fname}: {e}")
            wb.save()
            wb.close()
            print(f"Updated: {fname}")
        except Exception as e:
            print(f"Failed {fname}: {e}")
finally:
    app.quit()

Processing: CPT-1.xlsm
Updated: CPT-1.xlsm
Processing: CPTU_10.xlsm
Updated: CPTU_10.xlsm
Processing: CPTU_11.xlsm
Updated: CPTU_11.xlsm
Processing: CPTU_12.xlsm
Updated: CPTU_12.xlsm
Processing: CPTU_12_utvidet_plot.xlsm
Updated: CPTU_12_utvidet_plot.xlsm
Processing: CPTU_15.xlsm
Updated: CPTU_15.xlsm
Processing: CPTU_17.xlsm
Updated: CPTU_17.xlsm
Processing: CPTU_18.xlsm
Updated: CPTU_18.xlsm
Processing: CPTU_19.xlsm
Updated: CPTU_19.xlsm
Processing: CPTU_2.xlsm
Updated: CPTU_2.xlsm
Processing: CPTU_20.xlsm
Updated: CPTU_20.xlsm
Processing: CPTU_21.xlsm
Updated: CPTU_21.xlsm
Processing: CPTU_3.xlsm
Updated: CPTU_3.xlsm
Processing: CPTU_4.xlsm
Updated: CPTU_4.xlsm
Processing: CPTU_6.xlsm
Updated: CPTU_6.xlsm
Processing: CPTU_6_samlet.xlsm
Updated: CPTU_6_samlet.xlsm
Processing: CPTU_6_samlet_utvidet_plot.xlsm
Updated: CPTU_6_samlet_utvidet_plot.xlsm
Processing: CPTU_7.xlsm
Updated: CPTU_7.xlsm
Processing: CPTU_8.xlsm
Updated: CPTU_8.xlsm
Processing: CPTU_9.xlsm
Updated: CPTU_9.xlsm


In [4]:
from PyPDF2 import PdfFileMerger

pdf_dir = r"C:\Users\jdr\OneDriveMulti\Skrivebord\CPTU_Tåjeodden"
pdf_files = []

for fname in os.listdir(pdf_dir):
    if fname.lower().endswith(".xlsm") and not fname.startswith("~$"):
        base = os.path.splitext(fname)[0]
        profil_pdf = os.path.join(pdf_dir, f"{base}_Profil.pdf")
        cu_pdf = os.path.join(pdf_dir, f"{base}_Cu.pdf")
        if os.path.exists(profil_pdf) and os.path.exists(cu_pdf):
            pdf_files.append(profil_pdf)
            pdf_files.append(cu_pdf)

merger = PdfFileMerger()
for pdf in pdf_files:
    merger.append(pdf)

output_pdf = os.path.join(pdf_dir, "merged_profil_cu.pdf")
merger.write(output_pdf)
merger.close()
print(f"Merged PDF saved as: {output_pdf}")

Merged PDF saved as: C:\Users\jdr\OneDriveMulti\Skrivebord\CPTU_Tåjeodden\merged_profil_cu.pdf


Rev 270526

Oppdatere 8.SU med Nkt-faktor og Plaxis-SuA se 
https://multiconsultas.sharepoint.com/:x:/r/sites/10266411-01/Delte%20dokumenter/Oppdragsdokumenter/03_Arbeidsomr%C3%A5de/CPTU/C-profiler.xlsx?d=w122124af6711431aa578d165ccb5a74a&csf=1&web=1&e=AbPVku

In [3]:
# Oppdatere CPTU-filer med Cu-tolkning

from bisect import bisect_right
from math import floor
import os
import traceback
import xlwings as xw

cptu_path = r"C:\Users\jdr\Documents\CPTU_Tåjeodden\cpt_15"
pdf_path = os.path.join(cptu_path, "{}.pdf")
rev_field = "00"
rev_field_01 = "01"
date_str = "08.06.2026"
teg_nr = f"=Generelt!C37&\"-\"&Generelt!C42"
ocr_value = "OCR"  # Set your desired value for Generelt!C31
su_anbefalt_kurve_name = "Plaxis SuA"
p_values = [0.5, 1, 2, 4, 6, 10, 12, 15]
p_cells_su = [f"P{r}" for r in range(5, 17)]
q_cells_su = [f"Q{r}" for r in range(5, 17)]
clay_gamma = 17.5 # Defined for the project
plaxis_slam_depth_values = [0, 0.5, 1, 2, 3, 5, 10, 15, 20]
plaxis_slam_su_values = [1.632, 1.9395, 2.8245, 4.594, 6.025, 8.835, 15.85, 22.86, 30.0]
plaxis_3m_s_clay_depth_values = [3.5, 5.5, 8.5, 15, 20]
plaxis_3m_s_clay_su_values = [7.705, 12.765, 20.355, 36.785, 49.425]
plaxis_5m_s_clay_depth_values = [5.5, 15]
plaxis_5m_s_clay_su_values = [10.74, 34.77]
plaxis_7m_s_clay_depth_values = [7.5, 15]
plaxis_7m_s_clay_su_values = [12.53, 31.5]
plaxis_10m_s_clay_depth_values = [10.5, 15]
plaxis_10m_s_clay_su_values = [18.33, 29.71]
plaxis_clay_profiles = {
    3: (plaxis_3m_s_clay_depth_values, plaxis_3m_s_clay_su_values),
    5: (plaxis_5m_s_clay_depth_values, plaxis_5m_s_clay_su_values),
    7: (plaxis_7m_s_clay_depth_values, plaxis_7m_s_clay_su_values),
    10: (plaxis_10m_s_clay_depth_values, plaxis_10m_s_clay_su_values),
}

# p_values = [0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5, 5.5, 6, 6.5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 25]
# p_cells = [f"P{r}" for r in range(5, len(p_values) + 5)]
# q_cells = [f"Q{r}" for r in range(5, len(p_values) + 5)]


# def q_formula(p_cell_ref):
#     return f"=1+5/XLOOKUP({p_cell_ref},Beregn!A20:A2000,Beregn!U20:U2000)"

def interpolate_su(depth, depth_values, su_values):
    if depth <= depth_values[0]:
        return su_values[0]
    if depth >= depth_values[-1]:
        return su_values[-1]
    idx = bisect_right(depth_values, depth)
    x0, x1 = depth_values[idx - 1], depth_values[idx]
    y0, y1 = su_values[idx - 1], su_values[idx]
    if x1 == x0:
        return y0
    return y0 + (depth - x0) * (y1 - y0) / (x1 - x0)

def interpolate_or_extrapolate_su(depth, depth_values, su_values):
    if not depth_values or not su_values:
        return None
    if len(depth_values) == 1 or len(su_values) == 1:
        return su_values[0]
    if depth_values[0] < depth < depth_values[-1]:
        return interpolate_su(depth, depth_values, su_values)
    if depth <= depth_values[0]:
        x0, x1 = depth_values[0], depth_values[1]
        y0, y1 = su_values[0], su_values[1]
    else:
        x0, x1 = depth_values[-2], depth_values[-1]
        y0, y1 = su_values[-2], su_values[-1]
    if x1 == x0:
        return y1
    return y0 + (depth - x0) * (y1 - y0) / (x1 - x0)

def log_error(file_name, step, err):
    err_type = type(err).__name__
    tb = traceback.extract_tb(err.__traceback__)
    if tb:
        last = tb[-1]
        location = f"{os.path.basename(last.filename)}:{last.lineno} in {last.name}"
    else:
        location = "unknown"
    print(f"[ERROR] file='{file_name}' step='{step}' type='{err_type}' message='{err}' at {location}")

def try_set_com(file_name, step, setter):
    try:
        setter()
    except Exception as e:
        log_error(file_name, step, e)

def set_series_line_red(file_name, series_obj):
    # RGB(255,0,0) in Excel COM integer form
    red_color = 255
    try:
        series_obj.Format.Line.ForeColor.RGB = red_color
        return
    except Exception:
        pass
    try:
        series_obj.Border.Color = red_color
        return
    except Exception as e:
        log_error(file_name, "8.Cu chart set red line color", e)

def set_series_line_cyan(file_name, series_obj):
    # RGB(0,255,255) in Excel COM integer form
    # value = R + G*256 + B*256^2 = 0 + 255*256 + 255*65536 = 16776960
    cyan_color = 16776960
    try:
        series_obj.Format.Line.ForeColor.RGB = cyan_color
        return
    except Exception:
        pass
    try:
        series_obj.Border.Color = cyan_color
        return
    except Exception as e:
        log_error(file_name, "8.Cu chart set cyan line color", e)

def normalize_formula_text(text):
    if text is None:
        return ""
    return str(text).lower().replace("$", "").replace("'", "").replace(" ", "").replace(";", ",")

def find_series_by_tokens(chart_com, tokens):
    for i in range(1, chart_com.SeriesCollection().Count + 1):
        s = chart_com.SeriesCollection(i)
        ftxt = normalize_formula_text(getattr(s, 'Formula', ''))
        if all(t in ftxt for t in tokens):
            return s
    return None

def find_series_by_exact_name(chart_com, series_name):
    wanted = str(series_name).strip().lower()
    for i in range(1, chart_com.SeriesCollection().Count + 1):
        s = chart_com.SeriesCollection(i)
        try:
            current_name = str(s.Name).strip().lower()
            if current_name == wanted:
                return s
        except Exception:
            pass
    return None

def ensure_series(chart_com, name, x_values_formula, y_values_formula):
    chart_com.SeriesCollection().NewSeries()
    s = chart_com.SeriesCollection(chart_com.SeriesCollection().Count)
    s.Name = name
    s.XValues = x_values_formula
    s.Values = y_values_formula
    return s

def find_sheet_by_names(wb, candidate_names):
    wanted = [c.strip().lower() for c in candidate_names]
    try:
        for sh in wb.sheets:
            sh_name = str(sh.name).strip().lower()
            if sh_name in wanted:
                return sh
        for sh in wb.sheets:
            sh_name = str(sh.name).strip().lower()
            if any(w in sh_name for w in wanted):
                return sh
    except Exception:
        return None
    return None

def excel_sheet_ref(sheet_name):
    return "'" + str(sheet_name).replace("'", "''") + "'"

def get_max_depth_from_column(values):
    nums = []
    for v in values:
        if isinstance(v, list):
            v = v[0] if v else None
        if isinstance(v, (int, float)):
            nums.append(float(v))
    return max(nums) if nums else None

def select_clay_profile(depth_clay, profiles):
    default_m = min(profiles.keys())
    try:
        depth = float(depth_clay)
    except Exception:
        return default_m, profiles[default_m][0], profiles[default_m][1]

    selected_m = min(profiles.keys(), key=lambda m: abs(float(m) - depth))
    depth_values, su_values = profiles[selected_m]
    return selected_m, depth_values, su_values

app = xw.App(visible=False)  # Start Excel once

try:
    for fname in os.listdir(cptu_path):
        if not fname.lower().endswith(".xlsm"):
            continue
        fpath = os.path.join(cptu_path, fname)
        print(f"Processing: {fname}")
        try:
            wb = xw.Book(fpath)
            # Generelt!C31
            try:
                sht_gen = wb.sheets["Generelt"]
                # OCR value not pc as interpretation
                # sht_gen.range("C31").value = ocr_value
                # KS and godkjent initials
                # sht_gen.range("C13").value = "ASS"
                # sht_gen.range("C14").value = "ERIK-S"
                # Bp_num
                # c8_addr = sht_gen.range('C8').address
                # Teg-nr. Cu-plot
                # sht_gen.range("C42").value = f"={c8_addr}&\"-500.7\""
                # Rev and date 500.7 aka. Cu tolkning
                sht_gen.range("D42").number_format = "@"
                sht_gen.range("D42").value = rev_field_01
                sht_gen.range("E42").value = date_str
                # Date and revision fields 3. Profil
                sht_gen.range("E40").value = date_str
                sht_gen.range("D40").value = rev_field_01
                sht_gen.range("C30").value = su_anbefalt_kurve_name

            except Exception as e:
                log_error(fname, "Generelt update", e)
            # 3.Profil
            try:
                sht_prof = wb.sheets["3.Profil"]
                sht_prof.api.PageSetup.CenterFooter = ""
                sht_prof.api.PageSetup.LeftFooter = ""
                sht_prof.api.PageSetup.RightFooter = ""
                pdf_name = os.path.splitext(fname)[0]
                sht_prof.api.ExportAsFixedFormat(0, pdf_path.format(f"{pdf_name}_Profil"))
            except Exception as e:
                log_error(fname, "3.Profil export", e)
            # 7.OCR
            # try:
            #     sht_ocr = wb.sheets["7.OCR"]
            #     for cell_addr, val in zip(p_cells, p_values):
            #         sht_ocr.range(cell_addr).value = val
            #     for q_addr, p_addr in zip(q_cells, p_cells):
            #         formula = q_formula(p_addr)
            #         try:
            #             sht_ocr.range(q_addr).formula = formula
            #             # Check if the result is #N/A or None
            #             cell_value = sht_ocr.range(q_addr).value
            #             if cell_value is None or str(cell_value).strip().upper() == "#N/A":
            #                 sht_ocr.range(q_addr).value = ""
            #                 sht_ocr.range(p_addr).value = ""
            #         except Exception as e:
            #             print(f"Failed to write formula to {q_addr}: {e}")
            #     # Set Q3 value
            #     sht_ocr.range("Q3").value = "OCR4 Egendefinert"
            # except Exception as e:
            #     print(f"Error in {fname}: {e}")
            #     print(f"Skipping {fname}: sheet '7.OCR' not found")
         
            # Beregn
            max_depth = None
            beregn_sheet_name = "Beregn"
            sht_beregn = find_sheet_by_names(wb, ["Beregn", "Beregning"])
            try:
                if sht_beregn is not None:
                    beregn_sheet_name = str(sht_beregn.name)
                    max_depth = sht_beregn.range("A11").value
                else:
                    print(f"[WARN] file='{fname}' could not find Beregn sheet by name")
            except Exception as e:
                log_error(fname, "Beregn read max_depth", e)

            # Fallback if Beregn!A11 is unavailable/empty
            if max_depth is None:
                try:
                    if sht_beregn is not None:
                        a_vals = sht_beregn.range("A20:A10000").value
                        max_depth = get_max_depth_from_column(a_vals)
                except Exception as e:
                    log_error(fname, "Fallback read max_depth from Beregn!A20:A10000", e)

            if max_depth is None:
                max_depth = 20.0
                print(f"[WARN] file='{fname}' using default max_depth={max_depth}")

            plot_y_range = floor(float(max_depth) + 2)
               # 2.Spenn.
            try:
                sht_spenn = wb.sheets["2.Spenn."]
                depth_clay = sht_spenn.range("T6").value
                if depth_clay is None:
                    depth_clay = max_depth - 0.5
                elif sht_spenn.range("U6").value < clay_gamma:
                    for row in range(6, 20):
                        gamma_val = sht_spenn.range(f"U{row}").value
                        gamma_val_next = sht_spenn.range(f"U{row+1}").value
                        if gamma_val is not None and gamma_val >= clay_gamma and (gamma_val_next is None or gamma_val_next >= clay_gamma):
                            depth_clay = sht_spenn.range(f"T{row}").value
                            break
                print(f"[INFO] file='{fname}' depth_clay={depth_clay} max_depth={max_depth}")
                
            except Exception as e:
                log_error(fname, "2.Spenn. read depth_clay", e)
            # 6.N fakt
            try:
                n_fakt_sht = wb.sheets["6.N fakt"]
                n_fakt_sht.range("P21").value = 0
                n_fakt_sht.range("Q21").value = 6.5
                n_fakt_sht.range("P22").value = depth_clay
                n_fakt_sht.range("Q22").value = 6.5
                n_fakt_sht.range("P23").value = depth_clay + 0.01
                n_fakt_sht.range("Q23").value = 10
                
            except Exception as e:
                log_error(fname, "6.N fakt update", e)
            # 8.Cu
            try:
                sht_cu = wb.sheets["8.Cu"]
                sht_cu.range("AG5").value = 0.32
                sht_cu.range("AG6").value = 0.7
                sht_cu.range("J51").value = teg_nr
                sht_cu.range("K51").value = ""
                sht_cu.range("K51:M52").unmerge()
                sht_cu.range("J51:M52").merge()
                #sht_cu.range("J51:M52").horizontal_alignment = 'center'
                sht_cu.range("J51:M52").api.HorizontalAlignment = -4108  # xlCenter

                #sht_cu.range("J51").horizontal_alignment = 'center'
                depth_su_pairs = []

                # Slam segment up to depth_clay, including interpolation at boundary if needed
                for d, su in zip(plaxis_slam_depth_values, plaxis_slam_su_values):
                    if d <= depth_clay:
                        depth_su_pairs.append((d, su))
                if plaxis_slam_depth_values[0] <= depth_clay <= plaxis_slam_depth_values[-1] and depth_clay not in plaxis_slam_depth_values:
                    depth_su_pairs.append((depth_clay, interpolate_su(depth_clay, plaxis_slam_depth_values, plaxis_slam_su_values)))

                selected_clay_m, selected_clay_depth_values, selected_clay_su_values = select_clay_profile(depth_clay, plaxis_clay_profiles)
                print(f"[INFO] file='{fname}' selected clay profile={selected_clay_m}m")

                # Clay segment from just below depth_clay down to max_depth
                for d, su in zip(selected_clay_depth_values, selected_clay_su_values):
                    if depth_clay < d <= max_depth:
                        depth_su_pairs.append((d, su))
                if max_depth > depth_clay:
                    max_depth_su = interpolate_or_extrapolate_su(max_depth, selected_clay_depth_values, selected_clay_su_values)
                    if max_depth_su is not None:
                        depth_su_pairs.append((max_depth, max_depth_su))
                        if max_depth > selected_clay_depth_values[-1]:
                            print(f"[INFO] file='{fname}' extrapolated clay Su to max_depth={max_depth}")

                # Ensure sorted and remove potential duplicate depths
                dedup = {}
                for d, su in depth_su_pairs:
                    dedup[float(d)] = float(su)
                depth_su_pairs = sorted(dedup.items(), key=lambda x: x[0])

                sht_cu.range("P5:Q16").value = ""
                for (p_addr, q_addr), (depth_val, su_val) in zip(zip(p_cells_su, q_cells_su), depth_su_pairs):
                    sht_cu.range(p_addr).value = depth_val
                    sht_cu.range(q_addr).value = su_val
                
                # for chart in sht_cu.charts:
                #     print(chart)
                chart = sht_cu.charts[0]
                chart_obj = chart.api[0]
                # Get the actual Chart COM object
                chart_com = chart_obj.Chart
                # Set Y-axis limits from calculated depth range
                y_axis = chart_com.Axes(2)
                try_set_com(fname, "8.Cu chart y-axis MinimumScaleIsAuto", lambda: setattr(y_axis, 'MinimumScaleIsAuto', False))
                try_set_com(fname, "8.Cu chart y-axis MinimumScale", lambda: setattr(y_axis, 'MinimumScale', 0))
                try_set_com(fname, "8.Cu chart y-axis MaximumScaleIsAuto", lambda: setattr(y_axis, 'MaximumScaleIsAuto', False))
                try_set_com(fname, "8.Cu chart y-axis MaximumScale", lambda: setattr(y_axis, 'MaximumScale', plot_y_range))

                # Robustly find or create the two required series
                anbefalt_tokens = ["8.cu!q5:q120", "8.cu!p5:p120"]
                nkt_tokens = ["eg20:eg10000", "a20:a10000"]

                anbefalt_series = find_series_by_tokens(chart_com, anbefalt_tokens)
                if anbefalt_series is None:
                    try:
                        anbefalt_series = ensure_series(chart_com, "Plaxis SuA", "='8.Cu'!$Q$5:$Q$120", "='8.Cu'!$P$5:$P$120")
                    except Exception as e:
                        log_error(fname, "8.Cu create anbefalt_su series", e)

                nkt_series = find_series_by_exact_name(chart_com, "Nkt = Brukerdefinert")
                nkt_forced = False
                if nkt_series is None:
                    nkt_series = find_series_by_tokens(chart_com, nkt_tokens)
                if nkt_series is None:
                    try:
                        nkt_series = ensure_series(chart_com, "Nkt = Brukerdefinert", f"={excel_sheet_ref(beregn_sheet_name)}!$EG$20:$EG$10000", f"={excel_sheet_ref(beregn_sheet_name)}!$A$20:$A$10000")
                        nkt_forced = True
                    except Exception as e:
                        log_error(fname, "8.Cu create Nkt series", e)

                # Force style/visibility on every run
                if anbefalt_series is not None:
                    try_set_com(fname, "8.Cu chart anbefalt_su Line.Visible", lambda: setattr(anbefalt_series.Format.Line, 'Visible', True))
                    try_set_com(fname, "8.Cu chart anbefalt_su IsFiltered", lambda: setattr(anbefalt_series, 'IsFiltered', False))
                    try_set_com(fname, "8.Cu chart anbefalt_su Line.Weight", lambda: setattr(anbefalt_series.Format.Line, 'Weight', 1.5))
                    try_set_com(fname, "8.Cu chart anbefalt_su MarkerStyle", lambda: setattr(anbefalt_series, 'MarkerStyle', -4142))
                    set_series_line_red(fname, anbefalt_series)

                if nkt_series is not None:
                    # Only force Nkt styling/name if the series had to be created
                    if nkt_forced:
                        print(f"[INFO] file='{fname}' created Nkt series, applying name and style, color: cyan")
                        try_set_com(fname, "8.Cu chart Nkt Name", lambda: setattr(nkt_series, 'Name', 'Nkt = Brukerdefinert'))
                        try_set_com(fname, "8.Cu chart Nkt Line.Visible", lambda: setattr(nkt_series.Format.Line, 'Visible', True))
                        try_set_com(fname, "8.Cu chart Nkt IsFiltered", lambda: setattr(nkt_series, 'IsFiltered', False))
                        try_set_com(fname, "8.Cu chart Nkt Line.Weight", lambda: setattr(nkt_series.Format.Line, 'Weight', 1))
                        try_set_com(fname, "8.Cu chart Nkt MarkerStyle", lambda: setattr(nkt_series, 'MarkerStyle', -4142))
                        set_series_line_cyan(fname, nkt_series)
                # Export as PDF using file name (without extension)
                sht_cu.api.PageSetup.CenterFooter = ""
                sht_cu.api.PageSetup.LeftFooter = ""
                sht_cu.api.PageSetup.RightFooter = ""
                pdf_name = os.path.splitext(fname)[0]
                sht_cu.api.ExportAsFixedFormat(0, pdf_path.format(f"{pdf_name}_Cu"))
            except Exception as e:
                log_error(fname, "8.Cu update/chart/pdf", e)
            wb.save()
            wb.close()
            print(f"Updated: {fname}")
        except Exception as e:
            log_error(fname, "Open/save workbook", e)
finally:
    app.quit()

Processing: CPTU_15.xlsm
[WARN] file='CPTU_15.xlsm' could not find Beregn sheet by name
[WARN] file='CPTU_15.xlsm' using default max_depth=20.0
[INFO] file='CPTU_15.xlsm' depth_clay=12.4 max_depth=20.0
[INFO] file='CPTU_15.xlsm' selected clay profile=10m
[INFO] file='CPTU_15.xlsm' extrapolated clay Su to max_depth=20.0
Updated: CPTU_15.xlsm
